# Cropland latent regime MLE model

このノートブックは、世界共通の1本の回帰関数と、複数の潜在的な回帰関数を比較します。

現在のデータは単一時点の空間データなので、厳密なHMMではなく、**潜在レジーム付き有限混合モデル**を使います。

各レジームには、現在の研究設計に対応した二段階モデルを持たせます。

1. presence：農地が存在する確率
2. conditional fraction：農地が存在する場合の農地割合

EMアルゴリズムで最尤推定し、K=1, 2, 3, 4を比較します。K=1が世界共通モデル、K>1が複数の生産関数です。

注意：このノートブックはレジーム数の統計的な探索用です。Kが決まった後に、レジームごとのLightGBM・SHAP分析を追加します。


## 【セル1】データ読み込み・共通設定


In [1]:
from __future__ import annotations

import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)
from sklearn.neighbors import BallTree
from lightgbm import LGBMClassifier, LGBMRegressor

warnings.filterwarnings("ignore", category=RuntimeWarning)

ROOT = Path(r"C:\masterresearch\Comparative_advantage")
GAEZ_DIR = ROOT / "GAEZ"
HYDE_DIR = ROOT / "HYDE3.4"
CROPLAND_DIR = HYDE_DIR / "cropland_npys"
DIST_DIR = ROOT / "distance_to_cities"
GLOFAS_DIR = ROOT / "GloFAS" / "processed_5min"
FEATURE_CACHE = GAEZ_DIR / "CroplandRegression" / "features_cache"
OUTPUT_DIR = GAEZ_DIR / "CroplandRegression" / "spatial_wx_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
YEAR = 2024
PRESENCE_THRESHOLD = 0.01
N_POS_SAMPLE = 120_000
N_ZERO_SAMPLE = 120_000
N_SPLITS = 5

WX_RADII_KM = (50, 100)
WX_SOURCE_FEATURES = [
    "log_pop_density_2024",
    "log_city_time_20k_min",
    "log_port_time_any_min",
    "log_rainfed_value_top5",
    "log_glofas_p10_2020",
    "log_distance_river_gt10_2020",
]
INCLUDE_SLOPE_WX = False
MIN_VALID_NEIGHBOR_FRACTION = 0.25
SAVE_WX_RASTERS = True

EARTH_RADIUS_KM = 6371.0088
MORAN_K = 8
MORAN_MAX_N = 50_000


def load_cache(*names):
    for name in names:
        path = FEATURE_CACHE / name
        if path.exists():
            print("load cache:", path.name)
            return np.load(path, mmap_mode="r")
    raise FileNotFoundError(
        "Required cache not found: " + ", ".join(names)
    )


def safe_log1p(values):
    values = np.asarray(values, dtype=np.float32)
    values = np.where(np.isfinite(values) & (values > 0), values, 0.0)
    return np.log1p(values).astype(np.float32)


lat = np.load(CROPLAND_DIR / "lat.npy")
lon = np.load(CROPLAND_DIR / "lon.npy")
SHAPE = (len(lat), len(lon))

cropland_raw = np.load(
    CROPLAND_DIR / "cropland_fraction_1950_2024.npy",
    mmap_mode="r",
)
years = np.load(CROPLAND_DIR / "years.npy")
year_index = int(np.where(years == YEAR)[0][0])
cropland_2024 = np.asarray(cropland_raw[year_index], dtype=np.float32).copy()
cropland_2024[~np.isfinite(cropland_2024)] = np.nan

population_density_2024 = load_cache("population_density_2024.npy")
elevation_m = load_cache("elevation_5min.npy")
slope = load_cache("slope_5min.npy")
exclusion = load_cache("exclusion_5min_mode.npy")
city_time_20k_min = np.load(DIST_DIR / "cities_10_1_12deg_min.npy", mmap_mode="r")
port_time_any_min = np.load(DIST_DIR / "ports_05_1_12deg_min.npy", mmap_mode="r")
glofas_p10 = np.load(GLOFAS_DIR / "p10_discharge_max_5min_2020.npy", mmap_mode="r")
distance_river_gt10 = np.load(
    GLOFAS_DIR / "distance_to_reliable_river_p10_gt_10_m3s_km_5min_2020.npy",
    mmap_mode="r",
)
rainfed_value_top5 = load_cache(
    "rainfed_value_top5_usd_per_ha_checked_36crops.npy",
    "rainfed_value_top5_checked_36crops.npy",
)
irrigated_value_top5 = load_cache(
    "irrigated_value_top5_usd_per_ha_checked_36crops.npy",
    "irrigated_value_top5_checked_36crops.npy",
)
rainfed_calorie_top5 = load_cache(
    "rainfed_calorie_top5_kcal_per_ha_checked_36crops.npy",
    "rainfed_calorie_top5_checked_36crops.npy",
)
irrigated_calorie_top5 = load_cache(
    "irrigated_calorie_top5_kcal_per_ha_checked_36crops.npy",
    "irrigated_calorie_top5_checked_36crops.npy",
)

land_mask = (
    np.isfinite(elevation_m)
    & np.isfinite(cropland_2024)
    & np.isfinite(population_density_2024)
    & (population_density_2024 >= 0)
)
presence = land_mask & (cropland_2024 > PRESENCE_THRESHOLD)

rng = np.random.default_rng(RANDOM_SEED)
pos_flat = np.flatnonzero(presence.ravel())
zero_flat = np.flatnonzero((land_mask & ~presence).ravel())
pos_sample = rng.choice(pos_flat, min(N_POS_SAMPLE, len(pos_flat)), replace=False)
zero_sample = rng.choice(zero_flat, min(N_ZERO_SAMPLE, len(zero_flat)), replace=False)
sample_flat = np.concatenate([pos_sample, zero_sample])
rng.shuffle(sample_flat)
rows, cols = np.unravel_index(sample_flat, SHAPE)


def take(array):
    return np.asarray(array[rows, cols])


sample = pd.DataFrame({
    "row": rows.astype(np.int32),
    "col": cols.astype(np.int32),
    "lat": lat[rows].astype(np.float32),
    "lon": lon[cols].astype(np.float32),
    "cropland_fraction": take(cropland_2024).astype(np.float32),
    "presence": (take(cropland_2024) > PRESENCE_THRESHOLD).astype(np.uint8),
    "elevation_m": take(elevation_m).astype(np.float32),
    "slope": take(slope).astype(np.float32),
    "exclusion_class": np.nan_to_num(take(exclusion), nan=-1).astype(np.int16),
    "log_pop_density_2024": safe_log1p(take(population_density_2024)),
    "log_city_time_20k_min": safe_log1p(take(city_time_20k_min)),
    "log_port_time_any_min": safe_log1p(take(port_time_any_min)),
    "log_glofas_p10_2020": safe_log1p(take(glofas_p10)),
    "log_distance_river_gt10_2020": safe_log1p(take(distance_river_gt10)),
    "log_rainfed_value_top5": safe_log1p(take(rainfed_value_top5)),
    "log_rainfed_calorie_top5": safe_log1p(take(rainfed_calorie_top5)),
    "log_irrigation_value_gain_top5": safe_log1p(
        np.maximum(take(irrigated_value_top5) - take(rainfed_value_top5), 0)
    ),
    "log_irrigation_calorie_gain_top5": safe_log1p(
        np.maximum(take(irrigated_calorie_top5) - take(rainfed_calorie_top5), 0)
    ),
}).replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

sample["spatial_block"] = (
    np.floor((sample["lat"] + 90) / 10).astype(int) * 36
    + np.floor((sample["lon"] + 180) / 10).astype(int)
)

base_feature_cols = [
    "elevation_m",
    "slope",
    "exclusion_class",
    "log_pop_density_2024",
    "log_city_time_20k_min",
    "log_port_time_any_min",
    "log_glofas_p10_2020",
    "log_distance_river_gt10_2020",
    "log_rainfed_value_top5",
    "log_rainfed_calorie_top5",
    "log_irrigation_value_gain_top5",
    "log_irrigation_calorie_gain_top5",
]

print("grid:", SHAPE)
print("sample rows:", len(sample))
print("sample presence share:", float(sample["presence"].mean()))

load cache: population_density_2024.npy
load cache: elevation_5min.npy
load cache: slope_5min.npy
load cache: exclusion_5min_mode.npy
load cache: rainfed_value_top5_usd_per_ha_checked_36crops.npy
load cache: irrigated_value_top5_usd_per_ha_checked_36crops.npy
load cache: rainfed_calorie_top5_kcal_per_ha_checked_36crops.npy
load cache: irrigated_calorie_top5_kcal_per_ha_checked_36crops.npy
grid: (2160, 4320)
sample rows: 240000
sample presence share: 0.5


## 【セル2】既存WXキャッシュの読み込み

WXは再計算せず、保存済みの`.npy`だけを読み込みます。


In [2]:
# 既存のWXキャッシュを読み込むだけ
# キャッシュがない場合は、ここでは再計算せずエラーにする。

sample_rows = sample["row"].to_numpy(dtype=int)
sample_cols = sample["col"].to_numpy(dtype=int)
wx_feature_cols = []

for radius_km in WX_RADII_KM:
    for source_name in WX_SOURCE_FEATURES:
        wx_name = f"wx_{radius_km}km_{source_name}"
        cache_path = OUTPUT_DIR / f"{wx_name}.npy"

        if not cache_path.exists():
            raise FileNotFoundError(
                "WXキャッシュがありません。再計算は行わず停止します: "
                f"{cache_path}"
            )

        print("load WX cache:", cache_path.name)
        wx_raster = np.load(cache_path, mmap_mode="r")

        if wx_raster.shape != SHAPE:
            raise ValueError(
                f"WXキャッシュのshapeが不正です: {cache_path} "
                f"{wx_raster.shape} != {SHAPE}"
            )

        sample[wx_name] = np.asarray(
            wx_raster[sample_rows, sample_cols],
            dtype=np.float32,
        )
        wx_feature_cols.append(wx_name)

print("WX feature count:", len(wx_feature_cols))


load WX cache: wx_50km_log_pop_density_2024.npy
load WX cache: wx_50km_log_city_time_20k_min.npy
load WX cache: wx_50km_log_port_time_any_min.npy
load WX cache: wx_50km_log_rainfed_value_top5.npy
load WX cache: wx_50km_log_glofas_p10_2020.npy
load WX cache: wx_50km_log_distance_river_gt10_2020.npy
load WX cache: wx_100km_log_pop_density_2024.npy
load WX cache: wx_100km_log_city_time_20k_min.npy
load WX cache: wx_100km_log_port_time_any_min.npy
load WX cache: wx_100km_log_rainfed_value_top5.npy
load WX cache: wx_100km_log_glofas_p10_2020.npy
load WX cache: wx_100km_log_distance_river_gt10_2020.npy
WX feature count: 12


## 【セル3】分析サンプル・空間fold作成


In [3]:
sample = (
    sample.replace([np.inf, -np.inf], np.nan)
    .dropna(subset=base_feature_cols + wx_feature_cols)
    .reset_index(drop=True)
)
analysis_sample = sample.copy()
y_all = analysis_sample["presence"].to_numpy(dtype=np.uint8)
groups = analysis_sample["spatial_block"].to_numpy()

if analysis_sample["presence"].nunique() < 2:
    raise ValueError("Both presence classes are required.")
target_prior = float(presence.sum() / land_mask.sum())

feature_cols_by_model = {
    "baseline": base_feature_cols,
    "wx": base_feature_cols + wx_feature_cols,
}

try:
    cell_area_km2 = np.load(CROPLAND_DIR / "grid_area_km2.npy", mmap_mode="r")
    sample_area = np.asarray(
        cell_area_km2[
            analysis_sample["row"].to_numpy(int),
            analysis_sample["col"].to_numpy(int),
        ],
        dtype=float,
    )
    n_pos_pop = int(presence.sum())
    n_zero_pop = int(land_mask.sum() - presence.sum())
    n_pos_samp = int((y_all == 1).sum())
    n_zero_samp = int((y_all == 0).sum())
    area_weight = np.where(
        y_all == 1,
        n_pos_pop / max(n_pos_samp, 1),
        n_zero_pop / max(n_zero_samp, 1),
    ) * sample_area
except FileNotFoundError:
    print("grid_area_km2.npy not found; area weighting skipped.")
    area_weight = None

splits = list(
    GroupKFold(n_splits=N_SPLITS).split(
        analysis_sample,
        y_all,
        groups=groups,
    )
)
print("analysis rows:", len(analysis_sample))
print("spatial blocks:", analysis_sample["spatial_block"].nunique())
print("target prior:", target_prior)

analysis rows: 240000
spatial blocks: 291
target prior: 0.3658932992121897


## 【セル4】raw crop-potential特徴量の作成

crop-potentialの大きさはraw、周辺変数は既存のlog/WX仕様を使います。
raw WXも保存済みキャッシュから読み込みます。


In [4]:
# 1. 既存のanalysis_sampleをコピー
# ------------------------------------------------------------

raw_analysis_sample = analysis_sample.copy()

raw_rows = raw_analysis_sample["row"].to_numpy(dtype=int)
raw_cols = raw_analysis_sample["col"].to_numpy(dtype=int)


def sample_raster(array):
    return np.asarray(
        array[raw_rows, raw_cols],
        dtype=np.float32,
    )


def clean_positive_feature(values):
    """
    正の値だけをraw値として残す。
    非正値は0にする。
    同時に、存在フラグと欠測フラグも返す。
    """
    values = np.asarray(values, dtype=np.float32)

    finite = np.isfinite(values)
    exists = finite & (values > 0)
    missing = ~finite

    clean = np.where(
        exists,
        values,
        0.0,
    ).astype(np.float32)

    return (
        clean,
        exists.astype(np.uint8),
        missing.astype(np.uint8),
    )


# ------------------------------------------------------------
# 2. crop potentialのraw値を作成
# ------------------------------------------------------------

rainfed_value_sample = sample_raster(rainfed_value_top5)
irrigated_value_sample = sample_raster(irrigated_value_top5)

rainfed_calorie_sample = sample_raster(rainfed_calorie_top5)
irrigated_calorie_sample = sample_raster(irrigated_calorie_top5)


(
    rainfed_value_raw,
    rainfed_value_exists,
    rainfed_value_missing,
) = clean_positive_feature(rainfed_value_sample)

(
    rainfed_calorie_raw,
    rainfed_calorie_exists,
    rainfed_calorie_missing,
) = clean_positive_feature(rainfed_calorie_sample)


# ------------------------------------------------------------
# 3. 灌漑gainと符号付き差分を作成
# ------------------------------------------------------------

value_pair_valid = (
    np.isfinite(rainfed_value_sample)
    & np.isfinite(irrigated_value_sample)
)

calorie_pair_valid = (
    np.isfinite(rainfed_calorie_sample)
    & np.isfinite(irrigated_calorie_sample)
)


# 符号付き差分
irrigation_value_delta_signed = np.where(
    value_pair_valid,
    irrigated_value_sample - rainfed_value_sample,
    0.0,
).astype(np.float32)

irrigation_calorie_delta_signed = np.where(
    calorie_pair_valid,
    irrigated_calorie_sample - rainfed_calorie_sample,
    0.0,
).astype(np.float32)


# 非負gain
irrigation_value_gain_raw = np.maximum(
    irrigation_value_delta_signed,
    0.0,
).astype(np.float32)

irrigation_calorie_gain_raw = np.maximum(
    irrigation_calorie_delta_signed,
    0.0,
).astype(np.float32)


# gainの存在フラグ
irrigation_value_gain_exists = (
    value_pair_valid
    & (irrigation_value_delta_signed > 0)
).astype(np.uint8)

irrigation_calorie_gain_exists = (
    calorie_pair_valid
    & (irrigation_calorie_delta_signed > 0)
).astype(np.uint8)


# 欠測フラグ
irrigation_value_gain_missing = (
    ~value_pair_valid
).astype(np.uint8)

irrigation_calorie_gain_missing = (
    ~calorie_pair_valid
).astype(np.uint8)


# ------------------------------------------------------------
# 4. DataFrameに追加
# ------------------------------------------------------------

raw_analysis_sample[
    "rainfed_value_top5_raw"
] = rainfed_value_raw

raw_analysis_sample[
    "rainfed_calorie_top5_raw"
] = rainfed_calorie_raw

raw_analysis_sample[
    "irrigation_value_gain_top5_raw"
] = irrigation_value_gain_raw

raw_analysis_sample[
    "irrigation_calorie_gain_top5_raw"
] = irrigation_calorie_gain_raw


raw_analysis_sample[
    "rainfed_value_top5_exists"
] = rainfed_value_exists

raw_analysis_sample[
    "rainfed_calorie_top5_exists"
] = rainfed_calorie_exists

raw_analysis_sample[
    "irrigation_value_gain_top5_exists"
] = irrigation_value_gain_exists

raw_analysis_sample[
    "irrigation_calorie_gain_top5_exists"
] = irrigation_calorie_gain_exists


raw_analysis_sample[
    "irrigation_value_delta_top5_signed"
] = irrigation_value_delta_signed

raw_analysis_sample[
    "irrigation_calorie_delta_top5_signed"
] = irrigation_calorie_delta_signed


raw_analysis_sample[
    "rainfed_value_top5_missing"
] = rainfed_value_missing

raw_analysis_sample[
    "rainfed_calorie_top5_missing"
] = rainfed_calorie_missing

raw_analysis_sample[
    "irrigation_value_gain_top5_missing"
] = irrigation_value_gain_missing

raw_analysis_sample[
    "irrigation_calorie_gain_top5_missing"
] = irrigation_calorie_gain_missing


# ------------------------------------------------------------
# 5. 新しいcrop potential feature list
# ------------------------------------------------------------

non_crop_base_cols = [
    "elevation_m",
    "slope",
    "exclusion_class",
    "log_pop_density_2024",
    "log_city_time_20k_min",
    "log_port_time_any_min",
    "log_glofas_p10_2020",
    "log_distance_river_gt10_min",
]

# 実際の列名を確認
if "log_distance_river_gt10_min" not in raw_analysis_sample.columns:
    non_crop_base_cols[-1] = "log_distance_river_gt10_2020"


raw_crop_cols = [
    # rawの大きさ
    "rainfed_value_top5_raw",
    "rainfed_calorie_top5_raw",
    "irrigation_value_gain_top5_raw",
    "irrigation_calorie_gain_top5_raw",

    # 0か正値か
    "rainfed_value_top5_exists",
    "rainfed_calorie_top5_exists",
    "irrigation_value_gain_top5_exists",
    "irrigation_calorie_gain_top5_exists",

    # 灌漑による符号付き差分
    "irrigation_value_delta_top5_signed",
    "irrigation_calorie_delta_top5_signed",

    # 欠測フラグ
    "rainfed_value_top5_missing",
    "rainfed_calorie_top5_missing",
    "irrigation_value_gain_top5_missing",
    "irrigation_calorie_gain_top5_missing",
]


raw_crop_feature_cols = (
    non_crop_base_cols
    + raw_crop_cols
)


# ------------------------------------------------------------
# 6. raw版のcrop potential WXを作る
# ------------------------------------------------------------

# 既存のraw crop-potential WXキャッシュを読み込むだけ
# キャッシュがない場合は、ここでは再計算せずエラーにする。

raw_crop_wx_feature_cols = []

for radius_km in WX_RADII_KM:
    raw_wx_names = [
        f"wx_{radius_km}km_rainfed_value_top5_raw",
        f"wx_{radius_km}km_rainfed_value_exists_share",
    ]

    for wx_name in raw_wx_names:
        cache_path = OUTPUT_DIR / f"{wx_name}.npy"

        if not cache_path.exists():
            raise FileNotFoundError(
                "raw WXキャッシュがありません。再計算は行わず停止します: "
                f"{cache_path}"
            )

        print("load raw WX cache:", cache_path.name)
        wx_raster = np.load(cache_path, mmap_mode="r")

        if wx_raster.shape != SHAPE:
            raise ValueError(
                f"raw WXキャッシュのshapeが不正です: {cache_path} "
                f"{wx_raster.shape} != {SHAPE}"
            )

        raw_analysis_sample[wx_name] = np.asarray(
            wx_raster[raw_rows, raw_cols],
            dtype=np.float32,
        )
        raw_crop_wx_feature_cols.append(wx_name)


# 既存WXから、log版rainfed value WXだけ除外
non_crop_wx_feature_cols = [
    col
    for col in wx_feature_cols
    if "log_rainfed_value_top5" not in col
]


raw_crop_wx_feature_cols = (
    raw_crop_feature_cols
    + non_crop_wx_feature_cols
    + raw_crop_wx_feature_cols
)


feature_cols_raw_by_model = {
    "raw_crop": raw_crop_feature_cols,
    "raw_crop_wx": raw_crop_wx_feature_cols,
}


print()
print("raw_crop feature count:", len(raw_crop_feature_cols))
print("raw_crop_wx feature count:", len(raw_crop_wx_feature_cols))
print()
print("raw_crop features:")
print(raw_crop_feature_cols)
print()
print("raw_crop_wx features:")
print(raw_crop_wx_feature_cols)


load raw WX cache: wx_50km_rainfed_value_top5_raw.npy
load raw WX cache: wx_50km_rainfed_value_exists_share.npy
load raw WX cache: wx_100km_rainfed_value_top5_raw.npy
load raw WX cache: wx_100km_rainfed_value_exists_share.npy

raw_crop feature count: 22
raw_crop_wx feature count: 36

raw_crop features:
['elevation_m', 'slope', 'exclusion_class', 'log_pop_density_2024', 'log_city_time_20k_min', 'log_port_time_any_min', 'log_glofas_p10_2020', 'log_distance_river_gt10_2020', 'rainfed_value_top5_raw', 'rainfed_calorie_top5_raw', 'irrigation_value_gain_top5_raw', 'irrigation_calorie_gain_top5_raw', 'rainfed_value_top5_exists', 'rainfed_calorie_top5_exists', 'irrigation_value_gain_top5_exists', 'irrigation_calorie_gain_top5_exists', 'irrigation_value_delta_top5_signed', 'irrigation_calorie_delta_top5_signed', 'rainfed_value_top5_missing', 'rainfed_calorie_top5_missing', 'irrigation_value_gain_top5_missing', 'irrigation_calorie_gain_top5_missing']

raw_crop_wx features:
['elevation_m', 'slope

## 【セル5】潜在混合モデル用の設計行列と目的変数

presenceとpositive fractionを、レジームごとの尤度計算に使える形へ変換します。


In [5]:
from copy import deepcopy

from scipy.special import expit, logit, logsumexp
from scipy.stats import norm

from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import OneHotEncoder, StandardScaler


LATENT_MODEL_FEATURES = list(
    feature_cols_raw_by_model["raw_crop_wx"]
)

LATENT_CATEGORICAL_FEATURES = [
    col
    for col in ["exclusion_class"]
    if col in LATENT_MODEL_FEATURES
]

LATENT_NUMERIC_FEATURES = [
    col
    for col in LATENT_MODEL_FEATURES
    if col not in LATENT_CATEGORICAL_FEATURES
]

LATENT_EPS = 1e-6


def make_design_matrices(train_df, test_df):
    """Fit preprocessing on train only and transform train/test."""

    train_numeric = train_df[
        LATENT_NUMERIC_FEATURES
    ].to_numpy(dtype=np.float32)

    test_numeric = test_df[
        LATENT_NUMERIC_FEATURES
    ].to_numpy(dtype=np.float32)

    if not np.isfinite(train_numeric).all():
        raise ValueError("NaN/inf found in train features.")

    if not np.isfinite(test_numeric).all():
        raise ValueError("NaN/inf found in test features.")

    scaler = StandardScaler()

    train_numeric = scaler.fit_transform(
        train_numeric
    ).astype(np.float64)

    test_numeric = scaler.transform(
        test_numeric
    ).astype(np.float64)

    train_parts = [train_numeric]
    test_parts = [test_numeric]
    feature_names = list(LATENT_NUMERIC_FEATURES)

    if LATENT_CATEGORICAL_FEATURES:

        encoder_kwargs = {
            "handle_unknown": "ignore",
        }

        try:
            encoder = OneHotEncoder(
                sparse_output=False,
                **encoder_kwargs,
            )
        except TypeError:
            encoder = OneHotEncoder(
                sparse=False,
                **encoder_kwargs,
            )

        train_cat = encoder.fit_transform(
            train_df[
                LATENT_CATEGORICAL_FEATURES
            ].astype(str)
        ).astype(np.float64)

        test_cat = encoder.transform(
            test_df[
                LATENT_CATEGORICAL_FEATURES
            ].astype(str)
        ).astype(np.float64)

        train_parts.append(train_cat)
        test_parts.append(test_cat)

        feature_names.extend(
            encoder.get_feature_names_out(
                LATENT_CATEGORICAL_FEATURES
            ).tolist()
        )

    X_train = np.hstack(train_parts)
    X_test = np.hstack(test_parts)

    return X_train, X_test, feature_names


def make_targets(frame):
    y = frame[
        "cropland_fraction"
    ].to_numpy(dtype=float)

    d = frame[
        "presence"
    ].to_numpy(dtype=np.int8)

    positive = d == 1

    t = np.zeros(len(frame), dtype=float)
    t[positive] = logit(
        np.clip(
            y[positive],
            LATENT_EPS,
            1.0 - LATENT_EPS,
        )
    )

    return y, d, t


## 【セル6】二段階有限混合モデル（EM・最尤推定）

各状態のpresence確率とconditional fractionを別々に推定し、観測尤度から潜在状態の責任度を更新します。


In [6]:
class LatentTwoStageMixture:
    """
    Finite mixture of two-stage models.

    Stage 1:
        Bernoulli logistic regression for presence.

    Stage 2:
        Gaussian regression for logit-transformed positive fraction.

    The mixture weights are constant across observations in this first
    likelihood-based model. Posterior state probabilities are obtained
    after observing y. This is the cross-sectional analogue of a latent
    state model, not a temporal HMM.
    """

    def __init__(
        self,
        n_states,
        max_iter=15,
        n_init=2,
        tol=1e-4,
        ridge_alpha=10.0,
        sigma_floor=0.05,
        random_state=42,
    ):
        self.n_states = int(n_states)
        self.max_iter = int(max_iter)
        self.n_init = int(n_init)
        self.tol = float(tol)
        self.ridge_alpha = float(ridge_alpha)
        self.sigma_floor = float(sigma_floor)
        self.random_state = int(random_state)

    def _fit_components(self, X, d, t, gamma):
        self.pi_ = np.clip(
            gamma.mean(axis=0),
            LATENT_EPS,
            None,
        )
        self.pi_ = self.pi_ / self.pi_.sum()

        self.presence_models_ = []
        self.conditional_models_ = []
        self.sigmas_ = []

        positive = d == 1

        for state in range(self.n_states):

            state_weight = gamma[:, state]

            presence_model = LogisticRegression(
                C=1.0,
                solver="lbfgs",
                max_iter=250,
                random_state=self.random_state + state,
            )

            presence_model.fit(
                X,
                d,
                sample_weight=state_weight,
            )

            conditional_model = Ridge(
                alpha=self.ridge_alpha,
            )

            positive_weight = state_weight[positive]

            conditional_model.fit(
                X[positive],
                t[positive],
                sample_weight=positive_weight,
            )

            conditional_mean = conditional_model.predict(
                X[positive]
            )

            weighted_sse = np.sum(
                positive_weight
                * (t[positive] - conditional_mean) ** 2
            )

            sigma = np.sqrt(
                weighted_sse
                / max(positive_weight.sum(), 1e-12)
                + self.sigma_floor**2
            )

            self.presence_models_.append(
                presence_model
            )

            self.conditional_models_.append(
                conditional_model
            )

            self.sigmas_.append(
                float(sigma)
            )

        self.sigmas_ = np.asarray(
            self.sigmas_,
            dtype=float,
        )

    def component_log_likelihood(self, X, d, t):
        n = len(X)
        component_ll = np.zeros(
            (n, self.n_states),
            dtype=float,
        )

        positive = d == 1

        for state in range(self.n_states):

            probability = self.presence_models_[
                state
            ].predict_proba(X)[:, 1]

            probability = np.clip(
                probability,
                LATENT_EPS,
                1.0 - LATENT_EPS,
            )

            component_ll[:, state] = np.where(
                positive,
                np.log(probability),
                np.log(1.0 - probability),
            )

            if positive.any():
                conditional_mean = (
                    self.conditional_models_[
                        state
                    ].predict(X[positive])
                )

                component_ll[
                    positive,
                    state,
                ] += norm.logpdf(
                    t[positive],
                    loc=conditional_mean,
                    scale=self.sigmas_[state],
                )

        return component_ll

    def _snapshot(self):
        return {
            "pi": self.pi_.copy(),
            "presence_models": deepcopy(
                self.presence_models_
            ),
            "conditional_models": deepcopy(
                self.conditional_models_
            ),
            "sigmas": self.sigmas_.copy(),
        }

    def _restore(self, state):
        self.pi_ = state["pi"]
        self.presence_models_ = state[
            "presence_models"
        ]
        self.conditional_models_ = state[
            "conditional_models"
        ]
        self.sigmas_ = state["sigmas"]

    def _initial_gamma(self, d, t, init_number):
        n = len(d)

        if self.n_states == 1:
            return np.ones((n, 1), dtype=float)

        rng = np.random.default_rng(
            self.random_state + init_number
        )

        if init_number == 0:
            response_signature = np.column_stack(
                [d.astype(float), t]
            )

            labels = KMeans(
                n_clusters=self.n_states,
                n_init=5,
                random_state=self.random_state,
            ).fit_predict(response_signature)

            gamma = np.full(
                (n, self.n_states),
                0.02 / max(self.n_states - 1, 1),
                dtype=float,
            )

            gamma[
                np.arange(n),
                labels,
            ] = 0.98

            return gamma

        gamma = rng.dirichlet(
            np.ones(self.n_states),
            size=n,
        )

        return gamma

    def fit(self, X, d, t):
        best_loglik = -np.inf
        best_state = None

        for init_number in range(self.n_init):

            gamma = self._initial_gamma(
                d,
                t,
                init_number,
            )

            previous_loglik = -np.inf

            for iteration in range(self.max_iter):

                self._fit_components(
                    X,
                    d,
                    t,
                    gamma,
                )

                component_ll = (
                    self.component_log_likelihood(
                        X,
                        d,
                        t,
                    )
                )

                log_joint = (
                    component_ll
                    + np.log(self.pi_)[None, :]
                )

                loglik = float(
                    logsumexp(
                        log_joint,
                        axis=1,
                    ).sum()
                )

                gamma = np.exp(
                    log_joint
                    - logsumexp(
                        log_joint,
                        axis=1,
                    )[:, None]
                )

                if (
                    np.isfinite(previous_loglik)
                    and abs(loglik - previous_loglik)
                    < self.tol * (1.0 + abs(previous_loglik))
                ):
                    break

                previous_loglik = loglik

            self._fit_components(
                X,
                d,
                t,
                gamma,
            )

            final_component_ll = (
                self.component_log_likelihood(
                    X,
                    d,
                    t,
                )
            )

            final_log_joint = (
                final_component_ll
                + np.log(self.pi_)[None, :]
            )

            final_loglik = float(
                logsumexp(
                    final_log_joint,
                    axis=1,
                ).sum()
            )

            if final_loglik > best_loglik:
                best_loglik = final_loglik
                best_state = self._snapshot()

        if best_state is None:
            raise RuntimeError("EM failed to find a valid state.")

        self._restore(best_state)
        self.train_loglik_ = float(best_loglik)
        self.n_features_in_ = X.shape[1]

        return self

    def predict_expected_fraction(self, X):
        expected = np.zeros(len(X), dtype=float)

        for state in range(self.n_states):

            presence_probability = (
                self.presence_models_[state]
                .predict_proba(X)[:, 1]
            )

            conditional_logit_mean = (
                self.conditional_models_[state]
                .predict(X)
            )

            conditional_fraction = expit(
                conditional_logit_mean
            )

            expected += (
                self.pi_[state]
                * presence_probability
                * conditional_fraction
            )

        return np.clip(expected, 0.0, 1.0)

    def predict_loglikelihood(self, X, d, t):
        component_ll = self.component_log_likelihood(
            X,
            d,
            t,
        )

        return logsumexp(
            component_ll
            + np.log(self.pi_)[None, :],
            axis=1,
        )

    def posterior_state_probability(self, X, d, t):
        component_ll = self.component_log_likelihood(
            X,
            d,
            t,
        )

        log_joint = (
            component_ll
            + np.log(self.pi_)[None, :]
        )

        return np.exp(
            log_joint
            - logsumexp(
                log_joint,
                axis=1,
            )[:, None]
        )

    def parameter_count(self):
        p = self.n_features_in_
        per_state = (p + 1) + (p + 1) + 1
        return self.n_states * per_state + (self.n_states - 1)


## 【セル7】K=1〜4の空間OOF最尤比較

KごとにEMをtrain fold内だけで推定し、test foldのlog-likelihoodと最終cropland fractionを評価します。


In [7]:
def regression_metrics(y_true, y_pred, weights=None):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if weights is None:
        weights = np.ones(len(y_true), dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)

    valid = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
        & np.isfinite(weights)
        & (weights > 0)
    )

    y_true = y_true[valid]
    y_pred = y_pred[valid]
    weights = weights[valid]

    mean_true = np.average(y_true, weights=weights)
    residual = y_true - y_pred
    ss_res = np.sum(weights * residual**2)
    ss_tot = np.sum(weights * (y_true - mean_true) ** 2)

    return {
        "n": int(len(y_true)),
        "r2": float(1.0 - ss_res / ss_tot)
        if ss_tot > 0 else np.nan,
        "rmse": float(
            np.sqrt(
                np.average(
                    residual**2,
                    weights=weights,
                )
            )
        ),
        "mae": float(
            np.average(
                np.abs(residual),
                weights=weights,
            )
        ),
    }


LATENT_K_VALUES = (1, 2, 3, 4)
LATENT_EM_MAX_ITER = 15
LATENT_EM_N_INIT = 2

y_all_latent, d_all_latent, t_all_latent = (
    make_targets(raw_analysis_sample)
)

latent_weight_specs = {
    "unweighted_sample": None,
}

if area_weight is not None:
    latent_weight_specs[
        "area_weighted_reweighted"
    ] = np.asarray(area_weight, dtype=float)

latent_oof_results = {}
latent_fold_rows = []

for n_states in LATENT_K_VALUES:

    print()
    print("=" * 80)
    print(
        f"LATENT TWO-STAGE MIXTURE: K={n_states}"
    )
    print("=" * 80)

    predictions = np.full(
        len(raw_analysis_sample),
        np.nan,
        dtype=float,
    )

    test_loglik = np.full(
        len(raw_analysis_sample),
        np.nan,
        dtype=float,
    )

    fold_ids = np.full(
        len(raw_analysis_sample),
        -1,
        dtype=np.int16,
    )

    for fold_number, (train_idx, test_idx) in enumerate(
        splits,
        start=1,
    ):

        print(
            f"fold {fold_number}/{N_SPLITS}"
        )

        train_df = raw_analysis_sample.iloc[
            train_idx
        ]

        test_df = raw_analysis_sample.iloc[
            test_idx
        ]

        X_train, X_test, _ = make_design_matrices(
            train_df,
            test_df,
        )

        d_train = d_all_latent[train_idx]
        d_test = d_all_latent[test_idx]
        t_train = t_all_latent[train_idx]
        t_test = t_all_latent[test_idx]

        model = LatentTwoStageMixture(
            n_states=n_states,
            max_iter=LATENT_EM_MAX_ITER,
            n_init=LATENT_EM_N_INIT,
            random_state=RANDOM_SEED + fold_number * 100,
        )

        model.fit(
            X_train,
            d_train,
            t_train,
        )

        predictions[test_idx] = (
            model.predict_expected_fraction(
                X_test
            )
        )

        test_loglik[test_idx] = (
            model.predict_loglikelihood(
                X_test,
                d_test,
                t_test,
            )
        )

        fold_ids[test_idx] = fold_number

        for weighting, weights in latent_weight_specs.items():

            if weights is None:
                fold_weights = None
            else:
                fold_weights = weights[test_idx]

            metric = regression_metrics(
                y_all_latent[test_idx],
                predictions[test_idx],
                weights=fold_weights,
            )

            latent_fold_rows.append({
                "k": n_states,
                "fold": fold_number,
                "weighting": weighting,
                "r2": metric["r2"],
                "rmse": metric["rmse"],
                "mae": metric["mae"],
                "mean_test_loglik": float(
                    np.mean(
                        test_loglik[test_idx]
                    )
                ),
            })

        del X_train, X_test
        gc.collect()

    latent_oof_results[n_states] = {
        "prediction": predictions,
        "test_loglik": test_loglik,
        "fold": fold_ids,
    }


latent_fold_metrics_df = pd.DataFrame(
    latent_fold_rows
)

latent_global_rows = []

for n_states, result in latent_oof_results.items():

    for weighting, weights in latent_weight_specs.items():

        metric = regression_metrics(
            y_all_latent,
            result["prediction"],
            weights=weights,
        )

        if weights is None:
            mean_test_loglik = float(
                np.mean(result["test_loglik"])
            )
        else:
            mean_test_loglik = float(
                np.average(
                    result["test_loglik"],
                    weights=weights,
                )
            )

        latent_global_rows.append({
            "k": n_states,
            "weighting": weighting,
            "r2": metric["r2"],
            "rmse": metric["rmse"],
            "mae": metric["mae"],
            "mean_oof_loglik": mean_test_loglik,
        })


latent_global_metrics_df = pd.DataFrame(
    latent_global_rows
)

print()
print("=" * 80)
print("LATENT MIXTURE GLOBAL OOF RESULTS")
print("=" * 80)
print(
    latent_global_metrics_df.round(6).to_string(
        index=False
    )
)

print()
print("=" * 80)
print("LATENT MIXTURE FOLD RESULTS")
print("=" * 80)
print(
    latent_fold_metrics_df.round(6).to_string(
        index=False
    )
)



LATENT TWO-STAGE MIXTURE: K=1
fold 1/5
fold 2/5
fold 3/5
fold 4/5
fold 5/5

LATENT TWO-STAGE MIXTURE: K=2
fold 1/5
fold 2/5
fold 3/5
fold 4/5
fold 5/5

LATENT TWO-STAGE MIXTURE: K=3
fold 1/5
fold 2/5
fold 3/5
fold 4/5


KeyboardInterrupt: 

## 【セル8】全サンプルでのBIC・状態数候補の比較

BICは通常の独立観測の近似です。空間相関があるため、最終判断では空間OOF log-likelihoodも併記します。


In [ ]:
X_full, _, latent_feature_names = make_design_matrices(
    raw_analysis_sample,
    raw_analysis_sample,
)

full_latent_models = {}
bic_rows = []

for n_states in LATENT_K_VALUES:

    print(
        f"fit full-data latent model K={n_states}"
    )

    model = LatentTwoStageMixture(
        n_states=n_states,
        max_iter=LATENT_EM_MAX_ITER,
        n_init=LATENT_EM_N_INIT,
        random_state=RANDOM_SEED + 900 + n_states,
    )

    model.fit(
        X_full,
        d_all_latent,
        t_all_latent,
    )

    n_parameters = model.parameter_count()
    bic = (
        -2.0 * model.train_loglik_
        + n_parameters * np.log(len(X_full))
    )

    full_latent_models[n_states] = model

    bic_rows.append({
        "k": n_states,
        "train_loglik": model.train_loglik_,
        "n_parameters": n_parameters,
        "bic": bic,
        "mixing_weights": ", ".join(
            f"{value:.4f}"
            for value in model.pi_
        ),
    })

bic_df = pd.DataFrame(bic_rows)

selection_weighting = (
    "area_weighted_reweighted"
    if area_weight is not None
    else "unweighted_sample"
)

oof_selection = latent_global_metrics_df[
    latent_global_metrics_df["weighting"]
    == selection_weighting
].copy()

best_k_oof = int(
    oof_selection.loc[
        oof_selection["mean_oof_loglik"].idxmax(),
        "k",
    ]
)

best_k_bic = int(
    bic_df.loc[
        bic_df["bic"].idxmin(),
        "k",
    ]
)

print()
print("=" * 80)
print("BIC AND STATE-COUNT SELECTION")
print("=" * 80)
print(bic_df.round(4).to_string(index=False))
print()
print(
    f"OOF log-likelihood best K "
    f"({selection_weighting}): {best_k_oof}"
)
print(f"BIC best K: {best_k_bic}")


## 【セル9】最尤推定した潜在レジームの可視化

観測されたcropland fractionを使った事後状態確率を地図化します。これは、状態の地理的な分布を確認するための図です。


In [ ]:
import matplotlib.pyplot as plt


PLOT_K = best_k_bic
plot_model = full_latent_models[PLOT_K]

posterior = plot_model.posterior_state_probability(
    X_full,
    d_all_latent,
    t_all_latent,
)

most_likely_state = posterior.argmax(axis=1)
posterior_confidence = posterior.max(axis=1)

plot_df = raw_analysis_sample[
    [
        "row",
        "col",
        "lat",
        "lon",
        "cropland_fraction",
        "presence",
    ]
].copy()

plot_df["latent_state"] = most_likely_state
plot_df["posterior_confidence"] = posterior_confidence

print("posterior state counts:")
print(
    plot_df["latent_state"]
    .value_counts()
    .sort_index()
    .to_string()
)

print()
print("mean posterior probability by state:")
print(
    pd.DataFrame(
        posterior,
        columns=[
            f"state_{i}"
            for i in range(PLOT_K)
        ],
    ).mean().round(5).to_string()
)


plot_rng = np.random.default_rng(
    RANDOM_SEED + 1900
)
max_plot_n = 120_000

if len(plot_df) > max_plot_n:
    plot_idx = plot_rng.choice(
        len(plot_df),
        max_plot_n,
        replace=False,
    )
else:
    plot_idx = np.arange(len(plot_df))

plot_points = plot_df.iloc[plot_idx]

fig, axes = plt.subplots(
    1,
    2,
    figsize=(17, 6),
)

axes[0].scatter(
    plot_points["lon"],
    plot_points["lat"],
    c=plot_points["latent_state"],
    s=1.0,
    alpha=0.5,
    cmap="tab10",
    vmin=-0.5,
    vmax=PLOT_K - 0.5,
    linewidths=0,
)

axes[0].set_title(
    f"Posterior most-likely latent state (K={PLOT_K})"
)
axes[0].set_xlabel("longitude")
axes[0].set_ylabel("latitude")
axes[0].set_xlim(-180, 180)
axes[0].set_ylim(-60, 85)
axes[0].grid(alpha=0.25)

confidence_plot = axes[1].scatter(
    plot_points["lon"],
    plot_points["lat"],
    c=plot_points["posterior_confidence"],
    s=1.0,
    alpha=0.5,
    cmap="viridis",
    vmin=1.0 / PLOT_K,
    vmax=1.0,
    linewidths=0,
)

axes[1].set_title(
    "Posterior confidence of state assignment"
)
axes[1].set_xlabel("longitude")
axes[1].set_ylabel("latitude")
axes[1].set_xlim(-180, 180)
axes[1].set_ylim(-60, 85)
axes[1].grid(alpha=0.25)

fig.colorbar(
    confidence_plot,
    ax=axes[1],
    label="max posterior probability",
)

plt.tight_layout()
plt.show()


## 【セル10】結果の読み方

K>1が採用されるには、BICだけでなく、空間OOF log-likelihoodがK=1を安定して上回る必要があります。

Kが決まった後、各潜在状態の回帰関数をLightGBMで再推定し、状態別SHAPを行います。
